# 03 - Erro com pontos flutuantes
Vamos aprender sobre como usar os erros de ponto flutuante para resolução de problemas numéricos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana3
```

Faça o checkout nessa nova branch:

```bash
git checkout semana3
```

<hr />

## Atividade 1
A função exponencial natural pode ser definida pelo limite:
$$
e^x=\lim_{n\to\infty}\left(1+\frac{x}{n}\right)^n,
$$
mas também é dada pela **série de Maclaurin**:
$$
e^x=\sum_{n=0}^{\infty}\frac{x^n}{n!}
=1+\frac{x}{1!}+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

Implemente em **Python** o cálculo de $e^x$ pela série, interrompendo a soma quando o termo ficar menor que o limite prático de contribuição, usando a precisão de máquina como critério.

In [1]:
import math
import sys

def exp_series(x, atol=0.0):
    """ Aproxima e^x pela série de Maclaurin com critério de parada numérico. """
    eps = sys.float_info.epsilon
    s = 1.0
    term = 1.0
    n = 0
    tol_abs = max(atol, eps)

    while True:
        n += 1
        term *= x / n
        s += term
        if abs(term) < eps * abs(s) or abs(term) < tol_abs:
            break
        if n > 10_000:
            break
    return s, n, term

# Demonstração
for val in [1.0, 5.0, -2.0]:
    approx, nterms, last = exp_series(val)
    print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, termos={nterms})")

x=+1 -> e^x ≈ 2.718281828459046 (math.exp=2.718281828459045, termos=18)
x=+5 -> e^x ≈ 148.4131591025766 (math.exp=148.4131591025766, termos=33)
x=-2 -> e^x ≈ 0.1353352832366127 (math.exp=0.1353352832366127, termos=24)


## Atividade 2

Implemente:
$$
e^x\approx\left(1+\frac{x}{n}\right)^n
$$
com $n$ crescente, e:
1. Explique por que, para $x<0$ e $n$ muito grande, pode ocorrer **cancelamento catastrófico**;
2. Proponha um critério de parada numérico para encerrar o crescimento de $n$ sem perder precisão.

In [1]:
import math


def exp_by_limit(x, start=8, tolerance=1e-14, maximum=2**52):
    n = max(start, int(-x) + 1, 1)
    previous = None
    while n <= maximum:
        current = math.exp(n * math.log1p(x / n))
        if previous is not None:
            change = abs(current - previous)
            if change <= tolerance * max(1.0, abs(current)) or current == previous:
                return current, n
        previous = current
        n *= 2
    return previous, maximum


for value in (1.0, 5.0, -2.0):
    approximation, n = exp_by_limit(value)
    print(f"x={value:+g}: limite={approximation:.16g}, exp={math.exp(value):.16g}, n={n}")


x=+1: limite=2.718281828459026, exp=2.718281828459045, n=70368744177664
x=+5: limite=148.4131591025758, exp=148.4131591025766, n=2251799813685248
x=-2: limite=0.135335283236605, exp=0.1353352832366127, n=35184372088832


## Atividade 3

Para $|x|$ grande, use:
$$
e^x = \left(e^{m\cdot 2^{-k}}\right)^{2^k}, \quad
k = \left\lceil \log_2\!\left(\frac{|x|}{\theta}\right)\right\rceil, \quad m = \frac{x}{2^k}
$$
Calcule $e^{m}$ pela série (Ex. 1) e depois eleve ao quadrado $k$ vezes.

In [1]:
import math


def exp_with_scaling(x, theta=1.0):
    if x == 0:
        return 1.0, 0
    squarings = max(0, math.ceil(math.log2(abs(x) / theta)))
    reduced = x / (2 ** squarings)
    value, _, _ = exp_series(reduced)
    for _ in range(squarings):
        value *= value
    return value, squarings


for value in (10.0, -20.0, 50.0):
    approximation, squarings = exp_with_scaling(value)
    print(f"x={value:+g}: aproximação={approximation:.8e}, erro={abs(approximation-math.exp(value)):.3e}, quadraturas={squarings}")


x=+10: aproximação=2.20264658e+04, erro=2.183e-11, quadraturas=4
x=-20: aproximação=2.06115362e-09, erro=2.151e-23, quadraturas=5
x=+50: aproximação=5.18470553e+21, erro=6.501e+07, quadraturas=6


## Atividade 4

Use:
$$
\cos x=\sum_{n=0}^{\infty}(-1)^n\frac{x^{2n}}{(2n)!}
$$
com a recursão:
$$
t_{n+1}=t_n\cdot\frac{-x^2}{(2n+1)(2n+2)}
$$
Defina um critério de parada baseado em `epsilon` e compare o erro relativo para $x\in[-20,20]$ (200 pontos) contra `math.cos(x)`.

In [1]:
import math
import sys


def cos_by_series(x, tolerance=1e-15):
    angle = math.remainder(x, 2 * math.pi)
    total = 1.0
    term = 1.0
    n = 0
    while n < 10_000:
        term *= -(angle * angle) / ((2*n + 1) * (2*n + 2))
        updated = total + term
        n += 1
        if updated == total or abs(term) <= max(sys.float_info.epsilon, tolerance*abs(updated)):
            return updated, n
        total = updated
    raise RuntimeError("A série não convergiu.")


points = [-20 + 40*i/199 for i in range(200)]
relative_errors = []
for point in points:
    reference = math.cos(point)
    approximation, _ = cos_by_series(point)
    relative_errors.append(abs(approximation-reference) / max(abs(reference), sys.float_info.epsilon))

worst = max(range(len(points)), key=relative_errors.__getitem__)
print(f"Maior erro relativo: {relative_errors[worst]:.3e}, em x={points[worst]:.6f}")


Maior erro relativo: 2.781e-14, em x=4.723618


## Atividade 5

Dado $x$ e uma tolerância $\tau$, encontre o menor $N$ tal que:
$$
R_{N+1}(x)=\sum_{n=N+1}^{\infty}\frac{|x|^n}{n!} < \tau
$$

In [1]:
from decimal import Decimal, localcontext


def smallest_order(x, tolerance=1e-12, precision=80):
    if tolerance <= 0:
        raise ValueError("A tolerância deve ser positiva.")
    with localcontext() as context:
        context.prec = precision
        magnitude = Decimal(str(abs(x)))
        target = Decimal(str(tolerance))
        complete_sum = magnitude.exp()
        partial_sum = Decimal(1)
        term = Decimal(1)
        order = 0
        while complete_sum - partial_sum >= target:
            order += 1
            term *= magnitude / order
            partial_sum += term
        return order


for value in (1, 3, 10):
    print(f"x={value}: menor N = {smallest_order(value)}")


x=1: menor N = 14
x=3: menor N = 23
x=10: menor N = 46


## Atividade 6

Usando `decimal` ou `mpmath`, compute $e^x$ em alta precisão e compare com o resultado de `float64` (Ex. 1) para $x\in\{20, 40, 50\}$.
Analise:
- perda de dígitos significativos;
- quando o `float64` começa a saturar por overflow.


In [1]:
import math
import sys
from decimal import Decimal, localcontext


def decimal_exp(x, precision=80):
    with localcontext() as context:
        context.prec = precision
        return +Decimal(str(x)).exp()


for value in (20, 40, 50):
    high_precision = decimal_exp(value)
    floating_point = math.exp(value)
    relative_error = abs((Decimal.from_float(floating_point)-high_precision)/high_precision)
    print(f"x={value}: float64={floating_point:.16e}, erro relativo={relative_error:.3E}")

overflow_point = math.log(sys.float_info.max)
print(f"O float64 satura para x acima de aproximadamente {overflow_point:.12f}.")


x=20: float64=4.8516519540979028e+08, erro relativo=1.006E-18
x=40: float64=2.3538526683702000e+17, erro relativo=6.199E-17
x=50: float64=5.1847055285870720e+21, erro relativo=8.082E-17
O float64 satura para x acima de aproximadamente 709.782712893384.


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 3"
git push origin semana3
```